<a href="https://colab.research.google.com/github/nitshar002/real-estate-pricing-model/blob/main/modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Importing Advanced Preprocessing File

In [ ]:
user = 'zengtao' #This is whoever's advanced preprocessing you want to use.
user = user.lower()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

train_path = f"/content/drive/MyDrive/IDX Housing Data/{user}_AdvancedProcessingAndModeling/{user}_train_processed.parquet"
test_path = f"/content/drive/MyDrive/IDX Housing Data/{user}_AdvancedProcessingAndModeling/{user}_test_processed.parquet"

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

import os

if not os.path.exists(train_path):
    raise ValueError(
        f"No advanced preprocessing found for {USER}. "
        "Run advanced preprocessing notebook first."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Pre Modeling

In [ ]:
## Split features
X_train = train_df.drop(columns=['ClosePrice'])
y_train = train_df['ClosePrice']

X_test = test_df.drop(columns=['ClosePrice'])
y_test = test_df['ClosePrice']

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = train_df.drop(columns=['ClosePrice']).select_dtypes(include=['number']).columns

binary_cols = [
    c for c in num_cols
    if train_df[c].dropna().isin([0,1]).all()
]

scale_cols = [c for c in num_cols if c not in binary_cols]

scaler_X = StandardScaler(copy=False)

train_df[scale_cols] = scaler_X.fit_transform(
    train_df[scale_cols].astype('float32')
)

test_df[scale_cols] = scaler_X.transform(
    test_df[scale_cols].astype('float32')
)

# Modeling

## Linear regression test - R squared, MAPE and MdAPE

In [ ]:
unknown_cols = [col for col in train_df.columns if (train_df[col] == 'Unknown').any()]

print(f"Columns containing 'Unknown': {unknown_cols}")
# dropping HighSchoolDistrict for now.. to make sure everything is set up.
if 'HighSchoolDistrict' in unknown_cols:
  train_df = train_df.drop(columns=['HighSchoolDistrict'])
  test_df = test_df.drop(columns=['HighSchoolDistrict'])

Columns containing 'Unknown': []


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_percentage_error

# Create model
lr_model = LinearRegression()

# Fit model
lr_model.fit(X_train, y_train)

# Predict
y_pred = lr_model.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
mdape = np.median(np.abs((y_test - y_pred) / y_test))  # Manual MdAPE

print(f"R^2 Score: {r2:.4f}")
print(f"MAPE: {mape:.4f}")
print(f"MdAPE: {mdape:.4f}")

train_preds = lr_model.predict(X_train)
train_r2 = r2_score(y_train, train_preds)

train_mape = mean_absolute_percentage_error(y_train, train_preds)
train_mdape = np.median(np.abs((y_train - train_preds) / y_train))  # Manual MdAPE

print(f"\nTraining R^2 Score: {train_r2:.4f}")
print(f"MAPE: {train_mape:.4f}")
print(f"MdAPE: {train_mdape:.4f}")

R^2 Score: 0.8020
MAPE: 0.2703
MdAPE: 0.1679

Training R^2 Score: 0.8280
MAPE: 0.2214
MdAPE: 0.1580


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Fit model
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

# Metrics
r2_rf = r2_score(y_test, y_pred_rf)
mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf)
mdape_rf = np.median(np.abs((y_test - y_pred_rf) / y_test))

print(f"Random Forest R^2 Score: {r2_rf:.4f}")
print(f"Random Forest MAPE: {mape_rf:.4f}")
print(f"Random Forest MdAPE: {mdape_rf:.4f}")

train_preds = rf_model.predict(X_train)

train_r2 = r2_score(y_train, train_preds)
train_mape_rf = mean_absolute_percentage_error(y_train, train_preds)
train_mdape_rf = np.median(np.abs((y_train - train_preds) / y_train))

print(f"\nTraining R^2 Score: {train_r2:.4f}")
print(f"Training Random Forest MAPE: {train_mape_rf:.4f}")
print(f"Training Random Forest MdAPE: {train_mdape_rf:.4f}")

KeyboardInterrupt: 

# XGBoost test

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xgb

# Create model
xgb_model = xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05, random_state=42)

# Fit model
xgb_model.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb_model.predict(X_test)

# Metrics
r2_xgb = r2_score(y_test, y_pred_xgb)
mape_xgb = mean_absolute_percentage_error(y_test, y_pred_xgb)
mdape_xgb = np.median(np.abs((y_test - y_pred_xgb) / y_test))

print(f"XGBoost R^2 Score: {r2_xgb:.4f}")
print(f"XGBoost MAPE: {mape_xgb:.4f}")
print(f"XGBoost MdAPE: {mdape_xgb:.4f}")

train_preds_xgb = xgb_model.predict(X_train)

train_r2_xgb = r2_score(y_train, train_preds_xgb)
train_mape_xgb = mean_absolute_percentage_error(y_train, train_preds_xgb)
train_mdape_xgb = np.median(np.abs((y_train - train_preds_xgb) / y_train))

print(f"\nTraining R^2 Score: {train_r2_xgb:.4f}")
print(f"Training XGBoost MAPE: {train_mape_xgb:.4f}")
print(f"Training XGBoost MdAPE: {train_mdape_xgb:.4f}")

In [ ]:
# LightGBM Testing
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import numpy as np

# Create model
lgb_model = LGBMRegressor(
    n_estimators=300,
    max_depth=8,
    num_leaves=64,
    learning_rate=0.1,
    subsample=0.8,          # same as XGBoost subsample
    colsample_bytree=0.8,   # same as XGBoost colsample_bytree
    n_jobs=-1,
    random_state=42
)

# Fit model
lgb_model.fit(X_train, y_train)

# Predict
y_pred_lgb = lgb_model.predict(X_test)

# Metrics
r2_lgb = r2_score(y_test, y_pred_lgb)
mape_lgb = mean_absolute_percentage_error(y_test, y_pred_lgb)
mdape_lgb = np.median(np.abs((y_test - y_pred_lgb) / y_test))

print(f"LightGBM R^2 Score: {r2_lgb:.4f}")
print(f"LightGBM MAPE: {mape_lgb:.4f}")
print(f"LightGBM MdAPE: {mdape_lgb:.4f}")

# Training performance
train_preds_lgb = lgb_model.predict(X_train)

train_r2_lgb = r2_score(y_train, train_preds_lgb)
train_mape_lgb = mean_absolute_percentage_error(y_train, train_preds_lgb)
train_mdape_lgb = np.median(np.abs((y_train - train_preds_lgb) / y_train))

print(f"\nTraining R^2 Score: {train_r2_lgb:.4f}")
print(f"Training LightGBM MAPE: {train_mape_lgb:.4f}")
print(f"Training LightGBM MdAPE: {train_mdape_lgb:.4f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.107866 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3514
[LightGBM] [Info] Number of data points in the train set: 111588, number of used features: 38
[LightGBM] [Info] Start training from score 1145225.770759
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv

In [ ]:
importance = pd.Series(lgb_model.feature_importances_, index=X_train.columns)
print(importance.sort_values(ascending=False).head(15))

Postal_Code_Encoded        2178
Longitude                  1789
Latitude                   1751
LivingArea                 1415
LotSizeSquareFeet          1389
District_Avg_Price         1376
YearBuilt                  1231
DistNearestRestaurantMi    1126
Living_Area_to_Bedrooms     987
DaysOnMarket                949
Living_Area_per_Story       884
Monthly_HOA                 662
BathroomsTotalInteger       407
Bed_to_Bath                 347
Home_Age                    289
dtype: int32


Start Tyler

Exporting our best model (LightGBM) using pkl

In [ ]:
print(lgb_model.feature_names_in_)

['ViewYN' 'WaterfrontYN' 'BasementYN' 'PoolPrivateYN' 'Latitude'
 'Longitude' 'LivingArea' 'DaysOnMarket' 'AttachedGarageYN' 'ParkingTotal'
 'YearBuilt' 'BathroomsTotalInteger' 'BedroomsTotal' 'FireplaceYN'
 'Stories' 'MainLevelBedrooms' 'NewConstructionYN' 'GarageSpaces'
 'LotSizeSquareFeet' 'HasCarpet' 'HasVinyl' 'HasStone' 'HasBamboo'
 'HasConcrete' 'HasBrick' 'HasLaminate' 'HasTile' 'HasWood'
 'HasUnknownFlooring' 'Monthly_HOA' 'District_Avg_Price'
 'Postal_Code_Encoded' 'log_HOA' 'Home_Age' 'Bed_to_Bath'
 'Living_Area_to_Bedrooms' 'Living_Area_per_Story'
 'DistNearestRestaurantMi']


In [ ]:
# 1. Create X sample of first 10 rows in X_test
X_sample = X_test.head(10)[lgb_model.feature_names_in_]

# 2. Generate the predictions
predicted_prices = lgb_model.predict(X_sample)

# 3. Grab the actual true prices
actual_prices = y_test.head(10).values

# 4. Calculate the Absolute Percentage Error for each row
error_pct = np.abs(predicted_prices - actual_prices) / actual_prices

# 5. Put them side-by-side in a clean DataFrame
results_df = pd.DataFrame({
    'Actual Price': actual_prices,
    'Predicted Price': predicted_prices,
    'Difference ($)': predicted_prices - actual_prices,
    'Error (%)': error_pct
})

# 6. Format the numbers to look like real currency and percentages
formatted_results = results_df.style.format({
    'Actual Price': '${:,.0f}',
    'Predicted Price': '${:,.0f}',
    'Difference ($)': '${:,.0f}',
    'Error (%)': '{:.2%}'
})

# Display the table
formatted_results

,Actual Price,Predicted Price,Difference ($),Error (%)
0,"$2,975,000","$1,009,849","$-1,965,151",66.06%
1,"$720,000","$783,883","$63,883",8.87%
2,"$600,000","$659,065","$59,065",9.84%
3,"$3,636,725","$3,926,321","$289,596",7.96%
4,"$1,500,000","$1,098,498","$-401,502",26.77%
5,"$750,000","$1,017,067","$267,067",35.61%
6,"$3,175,000","$2,689,448","$-485,552",15.29%
7,"$1,100,000","$1,084,337","$-15,663",1.42%
8,"$1,560,000","$1,895,282","$335,282",21.49%
9,"$1,175,000","$1,172,074","$-2,926",0.25%


In [ ]:
# Grab the very first row (index 1) from our 10-row sample
idx = 0
first_house = X_sample.iloc[idx]

print("--- Data for the First House ---")
print(f"Actual Price: ${actual_prices[idx]:,.0f}")
print(f"Predicted Price: ${predicted_prices[idx]:,.0f}")
print(f"Error: {error_pct[idx]:.2%}\n")

# Convert the single row into a vertical DataFrame so it's easy to read
pd.DataFrame({'Feature Value': first_house})

--- Data for the First House ---
Actual Price: $2,975,000
Predicted Price: $1,009,849
Error: 66.06%



,Feature Value
ViewYN,False
WaterfrontYN,False
BasementYN,False
PoolPrivateYN,False
Latitude,37.87801
Longitude,-121.86846
LivingArea,1600.0
DaysOnMarket,0
AttachedGarageYN,True
ParkingTotal,0.0


In [ ]:
from pathlib import Path
import joblib

# 1. Define your save directory
save_dir = Path(f"/content/drive/MyDrive/IDX Housing Data/{user}_AdvancedProcessingAndModeling/models/")

# 2. Create the directory if it doesn't exist
save_dir.mkdir(parents=True, exist_ok=True)

# 3. Define the full file path
file_path = save_dir / "lgb_model_v1.pkl"

# 4. Dump the model
model_name = lgb_model
joblib.dump(model_name, file_path)

print(f"Model saved successfully to: {file_path.absolute()}")

Model saved successfully to: /content/drive/MyDrive/IDX Housing Data/zengtao_AdvancedProcessingAndModeling/models/lgb_model_v1.pkl


End Tyler